# Energy Efficient GPU Computing - Tutorial

## Hands-on: Measuring and Optimizing GPU Energy Efficiency

The notebook in the third hands-on has two parts:

1. **Measure**: measure the energy consumption of a GPU.
2. **Optimize**: tune the energy efficiency of a kernel using the *core clock frequency*.

## Part 1: Measuring GPU energy consumption

Before we can *optimize* for energy, we have to *measure* it.

We will use the on-chip sensors accessible through **NVML** to read reported power.

We'll compare two measurement methods, starting with a basic measurement.

As usual, we start with installing and importing the packages that we'll need.

In [ ]:
# Setup
%%capture
%pip install -q nvidia-ml-py

import time
import numpy as np
import torch
import seaborn as sns
from matplotlib import pyplot as plt
import pynvml
sns.set_theme(style="whitegrid")
pynvml.nvmlInit()
gpu = pynvml.nvmlDeviceGetHandleByIndex(0)


Next, we'll define a small test function that performs a matrix multiplication using torch. We use this function as our test workload and measure GPU power consumption using NVML while it's running.

In [ ]:
# Use a matrix multiply kernel as workload with random data.
A = torch.randn(2048, 2048, device="cuda")
B = torch.randn(2048, 2048, device="cuda")
def run_gemm(repeats=1):
    for _ in range(repeats):
        C = A @ B
    torch.cuda.synchronize()
run_gemm(5) # warm-up

print("Current GPU power =", pynvml.nvmlDeviceGetPowerUsage(gpu) / 1000, "W")

### Method 1: Simple Internal Measurement

Measure power of a single kernel run. Multiply by time to get total energy consumption ($E = P \times t$).

We repeat the experiment to show the differences in the results between each measurement.

In [ ]:
runs = 100

# Run the kernel once and estimate energy = power x time. Repeat 100x to see the spread.
estimates = []
watts = []
for i in range(runs):
    start = time.perf_counter() # Start timer.
    run_gemm(1) # Run the kernel once.
    seconds = time.perf_counter() - start # Stop timer.
    watts.append(pynvml.nvmlDeviceGetPowerUsage(gpu) / 1000) # Get NVML's standard (averaged) power reading.
    estimates.append(watts[-1] * seconds) # energy estimate in mJ

estimates = np.array(estimates)
print(f"Measured power = {[round(i) for i in watts]} Watt")
print(f"Average energy: {np.average(estimates)} stdev: {np.std(estimates)}")
sns.scatterplot(x=list(range(1, runs+1)), y=estimates, s=80)
plt.xlabel("Measurement run [#]")
plt.ylabel("Energy Consumption [J]")
plt.title(f"Same kernel measured {runs} times")
plt.show()

The energy estimates vary greatly and the power reading looks unexpectedly low. How does this happen?

* NVML reports a **time-averaged** power, only updated every tens of miliseconds (depending on the GPU). So, the previous quick kernel is too short to show up.

### Method 2: Improving Internal Measurements

#### **Assignment 1**
Increase the value of `repeat` to repeat the kernel several times to improve the reliability of NVML measurements. How do the energy measurements relate to what we saw before with `repeat=1`? What about the standard deviation?

#### **Assignment 2**
Replace `average_power()` with `instant_power()` to use NVML's instantaneous power measurements. Does this improve our measurements? Experiment with lowering the number for `repeat` again, does measuring instant power improve over averaged power measurements?

#### **Assignment 3**
Instead of measuring power and estimating energy consumption using ($E = P \times t$), we can use `nvmlDeviceGetTotalEnergyConsumption` to directly query energy consumption. Does this improve our energy measurements?

In [ ]:
runs = 100
# Assignment 1: Experiment with increasing repeat
repeat = 1

def average_power(gpu):
    """ Get NVML's averaged power reading in Watt. """
    return pynvml.nvmlDeviceGetPowerUsage(gpu) / 1000   # mW -> W

def instant_power(gpu):
    """ Get NVML's instantaneous power reading in Watt.  """
    (v,) = pynvml.nvmlDeviceGetFieldValues(gpu, [pynvml.NVML_FI_DEV_POWER_INSTANT])
    return v.value.uiVal / 1000   # mW -> W

estimates = []
for i in range(runs):
    start = time.perf_counter() # Start timer.
    run_gemm(repeat) # Run the kernel once.
    seconds = time.perf_counter() - start # Stop timer.
    # Assignment 2: Swap out average power for instant power
    watts.append(average_power(gpu))
    estimates.append(watts[-1] * seconds / repeat) # energy estimate in mJ

    # Assignment 3: measure energy consumption directly
    # before = pynvml.nvmlDeviceGetTotalEnergyConsumption(gpu) / 1000 # Read energy, mJ -> J.
    # run_gemm(repeat) # Run the kernel N times.
    # after = pynvml.nvmlDeviceGetTotalEnergyConsumption(gpu) / 1000 # Read energy, mJ -> J.
    # estimates.append((after - before) / repeat)

estimates = np.array(estimates)
print(f"Average energy: {np.average(estimates)} stdev: {np.std(estimates)}")
sns.scatterplot(x=list(range(1, runs+1)), y=estimates, s=80)
plt.ylim(bottom=0)
plt.xlabel("Measurement run [#]")
plt.ylabel("Energy Consumption [J]")
plt.title(f"Same kernel measured {runs} times")
plt.show()

## Part 2: Optimizing GPU optimizing clock frequencies

Unfortunately, setting the GPU clock frequencies requires root privileges and we do not have those on Colab. However, we can use a data set that we have collected earlier with Kernel Tuner and still see what impact on energy efficiency we can have when optimizing GPU clock frequencies.

We start by installing Kernel Tuner and downloading the data set. Kernel Tuner will use this data set to simulate the optimization process based on performance and energy data collected using an earlier run of Kernel Tuner.

In [ ]:
%%capture
%pip install kernel-tuner seaborn
# TODO: check if installing seaborn is needed on colab, also update notebook 2 if not needed

!wget -O GEMM_A100_cache.json.bz2 "https://github.com/KernelTuner/kernel_tuner_tutorial/blob/master/energy/data/GEMM_NVML_NVIDIA_A100-PCIE-40GB_freq_cache_fake_timings.json.bz2?raw=true"
!bunzip2 GEMM_A100_cache.json.bz2

Next, we import the required packages and set some defaults for plotting with seaborn and matplotlib:

In [ ]:
import kernel_tuner as kt
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = [15, 8]
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("paper", rc={"font.size":10,"axes.titlesize":9,"axes.labelsize":12})
sns.set(font_scale = 1.6)

### Clock Frequency Tuning Setup

In this notebook, we will simulate the auto-tuning process of a highly-tunable matrix multiplication kernel on an Nvidia A100 GPU. Most of the information that is needed for our simulation is stored already with the data file. However, as we will also experiment with changing the tunable parameters, we still define those below.

In particular, we are going to specify the GPU core clock frequencies that we would like to test. These are specified to Kernel Tuner with the special parameter name `nvml_gr_clock`.

In [ ]:
# Tunable parameters
tune_params = dict()

# Specify what GPU clock frequencies will be tested:
# GPU frequency in MHz, 690 is closest to the baseclock of 765
clock_frequencies = [330, 510, 690, 870, 1050, 1230, 1410]

# Tunable parameter names needed for the code
tune_param_keys = ["nvml_gr_clock", "MWG", "NWG", "KWG", "MDIMC", "NDIMC", "MDIMA", "NDIMB", "KWI", "VWM", "VWN", "STRM", "STRN", "SA", "SB", "PRECISION"]

Next, we define a function that wraps the call to Kernel Tuner, because we will be calling it multiple times in this notebook. The function returns the best configuration, according to the current optimization objective, found during the tuning run.

In [ ]:
def get_optimal_config(objective, strategy="genetic_algorithm", **kwargs):
    res_opt, env_opt = kt.tune_cache("../data/GEMM_A100_cache.json",
        objective=objective,
        strategy=strategy,
        strategy_options=None if strategy == "brute_force" else dict(max_fevals=200),
        quiet=True,
        **kwargs
    )
    return env_opt['best_config'], res_opt

def print_best(config):
    print("Best configuration found:")
    print(f"\nExecution time:\t{config['time']} ms\nEnergy:\t\t{config['nvml_energy']} Joule\n")

Now that we have our function to easily call tune_cache, let's have a look at tuning our matrix multiply kernel for different optimization objectives.

### Tuning for Time

We start by simply optimizing for the lowest possible kernel execution time.

In [ ]:
config_race_to_idle, res_race_to_idle = get_optimal_config("time")
config_race_to_idle['name'] = "race-to-idle (global)"

print_best(config_race_to_idle)

### Tuning for Time, then optimize clocks for best energy

The next step is to use our previous time-optimal configuration, and re-tune only the clock frequencies for energy efficiency. The idea behind this step is that we would like to use the best performing kernel configuration, but run it at the most energy-friendly clock frequency.

In [ ]:
# Take the parameters of the fastest configuration found and find the most energy friendly GPU clock frequency
tune_params = {k: [v] for k, v in config_race_to_idle.items() if k in tune_param_keys}
tune_params["nvml_gr_clock"] = clock_frequencies

# tune the clock frequencies for energy efficiency
config_race_to_idle_plus_clocks, res_race_to_idle_plus_clocks = get_optimal_config('GFLOPS/W', tune_params=tune_params, strategy='brute_force')
config_race_to_idle_plus_clocks['name'] = "race-to-idle + clocks"

print_best(config_race_to_idle_plus_clocks)

### Tuning for Energy

The final step is to directly tune the whole search space for energy efficiency. This optimization step attempts to find the configuration with the highest overall energy efficiency in the whole search space, for any combination of tunable parameter values including the GPU clock frequency.

In [ ]:
config_energy_to_solution, res_energy_to_solution = get_optimal_config("GFLOPS/W")
config_energy_to_solution['name'] = "energy-to-solution (global)"

print_best(config_energy_to_solution)

### Plotting the Results

We can now look at the results in terms of energy efficiency for each optimization strategy in the bar chart below:

In [ ]:
df = pd.DataFrame([config_race_to_idle, config_race_to_idle_plus_clocks, config_energy_to_solution])
sns.barplot(x=df.nvml_energy, y=df.name, orient='h', hue=df.name, legend=False)
plt.xlabel('Energy (J), lower is better')
plt.ylabel('')
plt.title('Lowest energy configuration')
plt.tight_layout()

Finally, we can also make a scatterplot to show the relation between energy and time of our three different optimization efforts.

In addition, we can see the difference in how Kernel Tuner optimizes for performance and energy efficiency, and how the optimization algorithm improves over time, by plotting the configurations that have been explored during the optimization process.

In [ ]:
# plot our three configurations
sns.scatterplot(x=df['GFLOPS/W'], y=df['GFLOP/s'], hue=df.name, s=250)

# plot the configurations tried when optimizing for time
df_time = pd.DataFrame(res_race_to_idle)
sns.scatterplot(x=df_time['GFLOPS/W'], y=df_time['GFLOP/s'], hue=df_time.index, alpha=0.6, palette=sns.light_palette("midnightblue", as_cmap=True), legend=False)

df_time_clocks = pd.DataFrame(res_race_to_idle_plus_clocks)
sns.scatterplot(x=df_time_clocks['GFLOPS/W'], y=df_time_clocks['GFLOP/s'], alpha=0.9, color="orange", legend=False)

# plot the configurations tried when optimizing for energy
df_energy = pd.DataFrame(res_energy_to_solution)
sns.scatterplot(x=df_energy['GFLOPS/W'], y=df_energy['GFLOP/s'], hue=df_energy.index, alpha=0.6, palette=sns.light_palette("forestgreen", as_cmap=True), legend=False)

# finishing touches to the plot
plt.xlabel('GFLOPs per Watt, higher is better')
plt.ylabel('GFLOPs per second, higher is better')
plt.title('Energy versus Time')
plt.legend(title='', loc="upper left")
plt.tight_layout()